In [2]:
from aocd import get_data
data = get_data(day=5, year=2024)
data[0:10]

'79|97\n51|8'

# part 1

Safety protocols clearly indicate that new pages for the safety manuals must be printed in a very specific order. The notation X|Y means that if both page number X and page number Y are to be produced as part of an update, page number X must be printed at some point before page number Y.

The Elf has for you both the page ordering rules and the pages to produce in each update (your puzzle input), but can't figure out whether each update has the pages in the right order.

For example:

```
47|53
97|13
97|61
97|47
75|29
61|13
75|53
29|13
97|29
53|29
61|53
97|53
61|29
47|13
75|47
97|75
47|61
75|61
47|29
75|13
53|13

75,47,61,53,29
97,61,53,29,13
75,29,13
75,97,47,61,53
61,13,29
97,13,75,29,47
```

ok, we have two inputs: rules and sequences of numbers ("updates")
* rules specify the order of pairs of numbers
* seq. of numbers where we have to check if they are ordered according to the rules

plan:
* parse updates
* for each update: create list of all possible pairs
* for each pair (2x number + index): find rule applying to these two numbers, check numbers are sorted acccording to rule
* classify updates as all_correct or not
* get middle number for all_correct updates, sum

In [6]:
data_test = """47|53
97|13
97|61
97|47
75|29
61|13
75|53
29|13
97|29
53|29
61|53
97|53
61|29
47|13
75|47
97|75
47|61
75|61
47|29
75|13
53|13

75,47,61,53,29
97,61,53,29,13
75,29,13
75,97,47,61,53
61,13,29
97,13,75,29,47"""
data_test[:10]

'47|53\n97|1'

In [20]:
rules, updates = [x.splitlines() for x in data_test.split("\n\n")]
rules[:10], updates[:10]

(['47|53',
  '97|13',
  '97|61',
  '97|47',
  '75|29',
  '61|13',
  '75|53',
  '29|13',
  '97|29',
  '53|29'],
 ['75,47,61,53,29',
  '97,61,53,29,13',
  '75,29,13',
  '75,97,47,61,53',
  '61,13,29',
  '97,13,75,29,47'])

next, implement:
* function to check a given pair of numbers against a rule
* function to walk through updates and generate all possible pairs

In [43]:
a=[1,23,4]
a.index(23)

1

In [55]:
numbers = [29, 13]
ranks = [1, 3]

def breaks_a_rule(numbers, ranks):
    for r in rules:
        num1, num2 = [int(s) for s in r.split("|")]
        if (num1 in numbers) and (num2 in numbers): 
            num1_rank, num2_rank = ranks[numbers.index(num1)], ranks[numbers.index(num2)]
            if num1_rank>num2_rank: return True
    return False
(numbers, ranks, breaks_a_rule(numbers, ranks))

([29, 13], [1, 3], False)

In [56]:
updates[:5]

['75,47,61,53,29', '97,61,53,29,13', '75,29,13', '75,97,47,61,53', '61,13,29']

In [70]:
# def validate_update(update)
update = updates[3]
pages = [int(x) for x in update.split(",")]
pairs = []
for i in range(len(pages)):
    for f in range(i+1, len(pages)):
        pair_numbers = [pages[i], pages[f]]
        pair_ranks = [i, f]
        pairs.append([pair_numbers, pair_ranks])
pairs, len(pairs)             

([[[75, 97], [0, 1]],
  [[75, 47], [0, 2]],
  [[75, 61], [0, 3]],
  [[75, 53], [0, 4]],
  [[97, 47], [1, 2]],
  [[97, 61], [1, 3]],
  [[97, 53], [1, 4]],
  [[47, 61], [2, 3]],
  [[47, 53], [2, 4]],
  [[61, 53], [3, 4]]],
 10)

In [71]:
len(pages)

5

ok that looks good, should be (num_el^2 - num_el) / 2 pairs

In [72]:
update = updates[3]

def get_pairs(update):
    pages = [int(x) for x in update.split(",")]
    pairs = []
    for i in range(len(pages)):
        for f in range(i+1, len(pages)):
            pair_numbers = [pages[i], pages[f]]
            pair_ranks = [i, f]
            pairs.append([pair_numbers, pair_ranks])
    return pairs
get_pairs(update)             

[[[75, 97], [0, 1]],
 [[75, 47], [0, 2]],
 [[75, 61], [0, 3]],
 [[75, 53], [0, 4]],
 [[97, 47], [1, 2]],
 [[97, 61], [1, 3]],
 [[97, 53], [1, 4]],
 [[47, 61], [2, 3]],
 [[47, 53], [2, 4]],
 [[61, 53], [3, 4]]]

In [75]:
pairs = get_pairs(update)
pairs

for numbers, ranks in pairs:
    print(numbers, ranks, breaks_a_rule(numbers, ranks))

[75, 97] [0, 1] True
[75, 47] [0, 2] False
[75, 61] [0, 3] False
[75, 53] [0, 4] False
[97, 47] [1, 2] False
[97, 61] [1, 3] False
[97, 53] [1, 4] False
[47, 61] [2, 3] False
[47, 53] [2, 4] False
[61, 53] [3, 4] False


ok, this seems to work for that specific bad update

```The fourth update, 75,97,47,61,53, is not in the correct order: it would print 75 before 97, which violates the rule 97|75.```

let's check for all the updates in the test data

In [76]:
data_test

'47|53\n97|13\n97|61\n97|47\n75|29\n61|13\n75|53\n29|13\n97|29\n53|29\n61|53\n97|53\n61|29\n47|13\n75|47\n97|75\n47|61\n75|61\n47|29\n75|13\n53|13\n\n75,47,61,53,29\n97,61,53,29,13\n75,29,13\n75,97,47,61,53\n61,13,29\n97,13,75,29,47'

In [87]:
rules, updates = [x.splitlines() for x in data_test.split("\n\n")]

for u in updates:
    pairs = get_pairs(update)
    # correct = True
    pairs_violators = [breaks_a_rule(numbers, ranks) for numbers, ranks in pairs]
        # if : correct = False
    update_correct = not any(pairs_violators)
    print(u, pairs, pairs_violators, update_correct)

75,47,61,53,29 [[[75, 97], [0, 1]], [[75, 47], [0, 2]], [[75, 61], [0, 3]], [[75, 53], [0, 4]], [[97, 47], [1, 2]], [[97, 61], [1, 3]], [[97, 53], [1, 4]], [[47, 61], [2, 3]], [[47, 53], [2, 4]], [[61, 53], [3, 4]]] [True, False, False, False, False, False, False, False, False, False] False
97,61,53,29,13 [[[75, 97], [0, 1]], [[75, 47], [0, 2]], [[75, 61], [0, 3]], [[75, 53], [0, 4]], [[97, 47], [1, 2]], [[97, 61], [1, 3]], [[97, 53], [1, 4]], [[47, 61], [2, 3]], [[47, 53], [2, 4]], [[61, 53], [3, 4]]] [True, False, False, False, False, False, False, False, False, False] False
75,29,13 [[[75, 97], [0, 1]], [[75, 47], [0, 2]], [[75, 61], [0, 3]], [[75, 53], [0, 4]], [[97, 47], [1, 2]], [[97, 61], [1, 3]], [[97, 53], [1, 4]], [[47, 61], [2, 3]], [[47, 53], [2, 4]], [[61, 53], [3, 4]]] [True, False, False, False, False, False, False, False, False, False] False
75,97,47,61,53 [[[75, 97], [0, 1]], [[75, 47], [0, 2]], [[75, 61], [0, 3]], [[75, 53], [0, 4]], [[97, 47], [1, 2]], [[97, 61], [1,

something's wrong :-((
let's check the two functions

In [96]:
def get_pairs(update):
    pages = [int(x) for x in update.split(",")]
    pairs = []
    for i in range(len(pages)):
        for f in range(i+1, len(pages)):
            pair_numbers = [pages[i], pages[f]]
            pair_ranks = [i, f]
            pairs.append([pair_numbers, pair_ranks])
    return pairs         

97,61,53,29,13 [[[97, 61], [0, 1]], [[97, 53], [0, 2]], [[97, 29], [0, 3]], [[97, 13], [0, 4]], [[61, 53], [1, 2]], [[61, 29], [1, 3]], [[61, 13], [1, 4]], [[53, 29], [2, 3]], [[53, 13], [2, 4]], [[29, 13], [3, 4]]]


In [97]:
updates[1], get_pairs(updates[1])    

('97,61,53,29,13',
 [[[97, 61], [0, 1]],
  [[97, 53], [0, 2]],
  [[97, 29], [0, 3]],
  [[97, 13], [0, 4]],
  [[61, 53], [1, 2]],
  [[61, 29], [1, 3]],
  [[61, 13], [1, 4]],
  [[53, 29], [2, 3]],
  [[53, 13], [2, 4]],
  [[29, 13], [3, 4]]])

In [98]:
updates[0], get_pairs(updates[0])    

('75,47,61,53,29',
 [[[75, 47], [0, 1]],
  [[75, 61], [0, 2]],
  [[75, 53], [0, 3]],
  [[75, 29], [0, 4]],
  [[47, 61], [1, 2]],
  [[47, 53], [1, 3]],
  [[47, 29], [1, 4]],
  [[61, 53], [2, 3]],
  [[61, 29], [2, 4]],
  [[53, 29], [3, 4]]])

ok, `get_pairs` works

In [102]:
rules

['47|53',
 '97|13',
 '97|61',
 '97|47',
 '75|29',
 '61|13',
 '75|53',
 '29|13',
 '97|29',
 '53|29',
 '61|53',
 '97|53',
 '61|29',
 '47|13',
 '75|47',
 '97|75',
 '47|61',
 '75|61',
 '47|29',
 '75|13',
 '53|13']

In [106]:
numbers = [61, 13]
ranks = [1, 3]

def breaks_a_rule(numbers, ranks):
    for r in rules:
        num1, num2 = [int(s) for s in r.split("|")]
        if (num1 in numbers) and (num2 in numbers): 
            num1_rank, num2_rank = ranks[numbers.index(num1)], ranks[numbers.index(num2)]
            if num1_rank>num2_rank: return True
    return False
(numbers, ranks, breaks_a_rule(numbers, ranks))

([61, 13], [1, 3], False)

hm this one also works

typo!

```
rules, updates = [x.splitlines() for x in data_test.split("\n\n")]

for ***u*** in updates:
    pairs = get_pairs(update)
    # correct = True
    pairs_violators = [breaks_a_rule(numbers, ranks) for numbers, ranks in pairs]
        # if : correct = False
    update_correct = not any(pairs_violators)
    print(u, pairs_violators, update_correct)
```

In [115]:
rules, updates = [x.splitlines() for x in data_test.split("\n\n")]

for update in updates:
    pairs = get_pairs(update)
    pairs_violators = [breaks_a_rule(numbers, ranks) for numbers, ranks in pairs]
    update_correct = not any(pairs_violators)
    print(update, update_correct)

75,47,61,53,29 True
97,61,53,29,13 True
75,29,13 True
75,97,47,61,53 False
61,13,29 False
97,13,75,29,47 False


ok that matches the test data result in the instructiosn

In [116]:
update

'97,13,75,29,47'

In [124]:
import math 

rules, updates = [x.splitlines() for x in data_test.split("\n\n")]

middle_num = []

for update in updates:
    pairs = get_pairs(update)
    pairs_violators = [breaks_a_rule(numbers, ranks) for numbers, ranks in pairs]
    update_correct = not any(pairs_violators)
    print(update, update_correct)
    if update_correct:
        update_arr = [int(x) for x in update.split(",")]
        if len(update_arr) % 2 == 0:
            raise Exception("wtf")
        else:
            middle_num.append(update_arr[math.floor(len(update_arr)/2)])
middle_num, sum(middle_num)

75,47,61,53,29 True
97,61,53,29,13 True
75,29,13 True
75,97,47,61,53 False
61,13,29 False
97,13,75,29,47 False


([61, 53, 29], 143)

works. let's put it all together, restart the kernel apply it to the real data

In [2]:
from aocd import get_data
data = get_data(day=5, year=2024)
data[0:50]

'79|97\n51|89\n51|74\n74|65\n74|63\n74|53\n41|94\n41|54\n41'

In [3]:
import math 

def get_pairs(update):
    pages = [int(x) for x in update.split(",")]
    pairs = []
    for i in range(len(pages)):
        for f in range(i+1, len(pages)):
            pair_numbers = [pages[i], pages[f]]
            pair_ranks = [i, f]
            pairs.append([pair_numbers, pair_ranks])
    return pairs  

def breaks_a_rule(numbers, ranks):
    for r in rules:
        num1, num2 = [int(s) for s in r.split("|")]
        if (num1 in numbers) and (num2 in numbers): 
            num1_rank, num2_rank = ranks[numbers.index(num1)], ranks[numbers.index(num2)]
            if num1_rank>num2_rank: return True
    return False

rules, updates = [x.splitlines() for x in data.split("\n\n")]
middle_num = []

for update in updates:
    pairs = get_pairs(update)
    pairs_violators = [breaks_a_rule(numbers, ranks) for numbers, ranks in pairs]
    update_correct = not any(pairs_violators)
    # print(update, update_correct)
    if update_correct:
        update_arr = [int(x) for x in update.split(",")]
        if len(update_arr) % 2 == 0:
            raise Exception("wtf")
        else:
            middle_num.append(update_arr[math.floor(len(update_arr)/2)])

len(middle_num), sum(middle_num)

(99, 5732)

In [5]:
len(rules), len(updates)

(1176, 192)

ok that was stupid, there's a lot more rules than updates

# part 2 

For each of the incorrectly-ordered updates, use the page ordering rules to put the page numbers in the right order. For the above example, here are the three incorrectly-ordered updates and their correct orderings:

```
75,97,47,61,53 becomes 97,75,47,61,53.
61,13,29 becomes 61,29,13.
97,13,75,29,47 becomes 97,75,47,29,13.
After taking only the incorrectly-ordered updates and ordering them correctly, their middle page numbers are 47, 29, and 47. Adding these together produces 123.
```

ok now we have to sort the *incorrect* ones

In [ ]:
since this is about time and my solution is pretty ugly, let's do something even more stupid:
* for each incorrect update, find the broken rules
* for each broken rule, flip the numbers and check again - try maybe ten times?

In [7]:
data_test = """47|53
97|13
97|61
97|47
75|29
61|13
75|53
29|13
97|29
53|29
61|53
97|53
61|29
47|13
75|47
97|75
47|61
75|61
47|29
75|13
53|13

75,47,61,53,29
97,61,53,29,13
75,29,13
75,97,47,61,53
61,13,29
97,13,75,29,47"""
data_test[:10]

'47|53\n97|1'

In [43]:
import math 

rules, updates = [x.splitlines() for x in data_test.split("\n\n")]
middle_num = []

for update in updates:
    pairs = get_pairs(update)
    pairs_violators = [breaks_a_rule(numbers, ranks) for numbers, ranks in pairs]
    update_correct = not any(pairs_violators)

    if not update_correct:
        tries = 0
        while not update_correct and tries<100:
            tries += 1 
            # flip pair of violating elements
            for violatorIdx in [i for i,p in enumerate(pairs_violators) if p]:
                idx1, idx2 = pairs[violatorIdx][1]
                updateArr = [int(x) for x in update.split(",")]
                updateArr[idx1], updateArr[idx2] = updateArr[idx2], updateArr[idx1]
            updateCorrected = ",".join([str(i) for i in updateArr])
        
            # check again
            pairs = get_pairs(updateCorrected)
            pairs_violators = [breaks_a_rule(numbers, ranks) for numbers, ranks in pairs]
            # print(pairs_violators)
            update_correct = not any(pairs_violators)
        print(update, updateCorrected, tries)


75,97,47,61,53 97,75,47,61,53 1
61,13,29 61,29,13 1
97,13,75,29,47 97,13,75,47,29 100


ok, this is plain stupid and doesnt even work

* find all the rules that apply to a given update
* 

In [44]:
rules, updates = [x.splitlines() for x in data_test.split("\n\n")]
rules

['47|53',
 '97|13',
 '97|61',
 '97|47',
 '75|29',
 '61|13',
 '75|53',
 '29|13',
 '97|29',
 '53|29',
 '61|53',
 '97|53',
 '61|29',
 '47|13',
 '75|47',
 '97|75',
 '47|61',
 '75|61',
 '47|29',
 '75|13',
 '53|13']

In [52]:
rules, updates = [x.splitlines() for x in data_test.split("\n\n")]
# rules = [(int(num1), int(num2)) for r in rules for num1, num2 in r.split("|")]
rules[:5]

['47|53', '97|13', '97|61', '97|47', '75|29']

got this suggestion from mistral

```
def compare(a, b, rules):
    if (a, b) in rules:
        return -1
    elif (b, a) in rules:
        return 1
    else:
        return 0

def apply_rules_and_sort(numbers, rules):
    # Convert the rules into a set for faster lookup
    rules_set = set(rules)

    # Use the sorted function with a custom key
    sorted_numbers = sorted(numbers, key=lambda x: sorted(numbers, key=lambda y: compare(x, y, rules_set)))
    return sorted_numbers

# Example usage
numbers = [3, 1, 4, 2]
rules = [(1, 2), (2, 4), (3, 1), (3, 2)]

sorted_numbers = apply_rules_and_sort(numbers, rules)
print(f"Sorted numbers according to the rules: {sorted_numbers}")
```

In [60]:
?sorted

Signature: sorted(iterable, /, *, key=None, reverse=False)
Docstring:
Return a new list containing all items from the iterable in ascending order.

A custom key function can be supplied to customize the sort order, and the
reverse flag can be set to request the result in descending order.
Type:      builtin_function_or_method

ok, let's do it this way.
* transform the rules to tuples
* sort updates with `sorted` + a custom compare function

In [122]:
rules, updates = [x.splitlines() for x in data_test.split("\n\n")]

# rules to tuples, updates to arrays of numbers
rules = [(int(s[0]), int(s[1])) for s in [r.split("|") for r in rules]]
updates = [[int(v) for v in u.split(",")] for u in updates]

(rules[:5], updates[:5])

([(47, 53), (97, 13), (97, 61), (97, 47), (75, 29)],
 [[75, 47, 61, 53, 29],
  [97, 61, 53, 29, 13],
  [75, 29, 13],
  [75, 97, 47, 61, 53],
  [61, 13, 29]])

In [127]:
# test custom sort
u = updates[0]
pair = (u[0], u[1])
(pair, pair in rules, rules[rules.index(pair)])

((75, 47), True, (75, 47))

In [141]:
from functools import cmp_to_key

def compare(a, b): return -1 if (a, b) in rules else 1 if (b, a) in rules else 0

u = updates[0]
(u, sorted(u, key=cmp_to_key(compare)))

([75, 47, 61, 53, 29], [75, 47, 61, 53, 29])

In [142]:
sum = 0 
for u in updates:
    u_sorted = sorted(u, key=cmp_to_key(compare))
    is_sorted = u == u_sorted
    if not is_sorted:
        if len(update_arr) % 2 == 0: raise Exception("wtf")
        middle_num = u_sorted[math.floor(len(u)/2)]
        sum += middle_num
        # print(u, is_sorted, middle_num)
sum

123

ok this is finally correct for the test set. let's put it all together and push the real data through

## moment of truth

In [145]:
rules, updates = [x.splitlines() for x in get_data(day=5, year=2024).split("\n\n")]

# rules to tuples, updates to arrays of numbers
rules = [(int(s[0]), int(s[1])) for s in [r.split("|") for r in rules]]
updates = [[int(v) for v in u.split(",")] for u in updates]

# custom compare
def compare(a, b): return -1 if (a, b) in rules else 1 if (b, a) in rules else 0

sum = 0 
for u in updates:
    u_sorted = sorted(u, key=cmp_to_key(compare))
    is_sorted = u == u_sorted
    if not is_sorted:
        if len(update_arr) % 2 == 0: raise Exception("wtf")
        middle_num = u_sorted[math.floor(len(u)/2)]
        sum += middle_num
        # print(u, is_sorted, middle_num)
sum

4716

# 🎉🎉🥳